# Problem Set: Article Named-Entity Recognition (NER) Service
## Pipeline Pembuatan, Profiling Data, & Fine-Tuning Model spaCy
**Role:** Senior AI/NLP Engineer & Lead Fullstack Developer  
**Domain:** News-Monitoring (Media Online Malaysia)  
**Dataset:** `dataset/` (8 File CSV Ekspor Berita Media Daring)

---

### Daftar Isi Pipeline:
1. **Instalasi Dependensi & Setup Environment**
2. **Import Library & Konfigurasi Global**
3. **Pemuatan Data & Data Profiling (Dokumentasi Lengkap)**
4. **Pembersihan Teks & Pra-Pemrosesan (Deterministic Offset Rule)**
5. **Pembuatan Dataset Evaluasi Terverifikasi (`data/evaluation_examples.json`)**
6. **Bootstrap Anotasi Lemah (Weak Supervision) & Kamus Domain Berita**
7. **Pemisahan Data (70% Train, 15% Dev, 15% Test) Berbasis ID Artikel**
8. **Konversi ke Format Biner spaCy DocBin (`.spacy`)**
9. **Konfigurasi Model & Hyperparameter spaCy v3**
10. **Pelatihan Model (Fine-Tuning Loop)**
11. **Evaluasi Metrik: Baseline (`en_core_web_sm`) vs Fine-Tuned (`model/model-best`)**
12. **Analisis Error (False Positives & False Negatives)**
13. **Jawaban Lengkap Pertanyaan Teoretis & Etis Studi Kasus**


## 1. Instalasi Dependensi & Setup Environment

Pada tahap awal, kita menginstal pustaka yang dibutuhkan:
- `spacy`: Ekosistem NLP industri untuk tokenisasi, tok2vec, dan transition-based NER.
- `scikit-learn`: Untuk pemisahan dataset secara deterministik (`train_test_split`).
- `pandas`: Untuk memanipulasi dan memprofil dataset tabular CSV.
- `fastapi`, `uvicorn`, `httpx`, `pytest`: Untuk backend API dan suite pengujian.
- Model bahasa dasar spaCy: `en_core_web_sm`.


In [1]:
# Instalasi dependensi (Jalankan cell ini jika dependensi belum terpasang)
!pip install -q spacy pandas scikit-learn fastapi uvicorn httpx pytest nbformat
!python -m spacy download en_core_web_sm


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --- ------------------------------------ 1.0/12.8 MB 8.4 MB/s eta 0:00:02
     --------- ------------------------------ 2.9/12.8 MB 8.8 MB/s eta 0:00:02
     -------------- ------------------------- 4.7/12.8 MB 8.8 MB/s eta 0:00:01
     -------------------- ------------------- 6.6/12.8 MB 8.9 MB/s eta 0:00:01
     --------------------------- ------------ 8.7/12.8 MB 8.9 MB/s eta 0:00:01
     -------------------------------- ------- 10.5/12.8 MB 8.9 MB/s eta 0:00:01
     ------------------------------------ --- 11.8/12.8 MB 8.7 MB/s eta 0:00:01
     ---------------------------------------  12.6/12.8 MB 8.5 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 8.0 MB/s  0:00:01
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


## 2. Import Library & Konfigurasi Global

Mengimpor pustaka utama Python, menetapkan seed acak untuk menjamin reprodusibilitas hasil (deterministik), dan menetapkan target kategori entitas:
- `PERSON`: Tokoh, pemimpin negara, atlet, eksekutif (misal: *Anwar Ibrahim*, *Tan Sri Mohd Dusuki Mokhtar*).
- `ORG`: Lembaga, perbankan, perusahaan, asosiasi (misal: *Maybank*, *CIMB Group*, *Bank Negara Malaysia*, *MHL*).
- `GPE`: Negara, kota, entitas geopolitik (misal: *Malaysia*, *Kuala Lumpur*, *Putrajaya*).
- `LOC`: Lokasi geografis non-GPE (misal: *Carey Island*, *Port Dickson*).
- `DATE`: Tanggal, bulan, tahun (misal: *8 November*, *October 2026*).
- `EVENT`: Acara, festival, perayaan (misal: *Deepavali Day*, *Financial Lines Forum*).
- `NORP`: Kebangsaan, kelompok etnis, afiliasi politik (misal: *Malaysian*, *Rohingya*).
- `MONEY`: Nilai mata uang (misal: *RM10.7 billion*, *RM300,000*).


In [2]:
import os
import re
import glob
import json
import random
import subprocess
from pathlib import Path

import pandas as pd
import spacy
from spacy.tokens import DocBin
from spacy.util import filter_spans
from spacy.scorer import Scorer
from spacy.training import Example
from sklearn.model_selection import train_test_split

# Kunci determinisme
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Kategori target entitas
TARGET_LABELS = {"PERSON", "ORG", "GPE", "LOC", "DATE", "EVENT", "NORP", "MONEY"}

# Path direktori
BASE_DIR = Path(".").resolve()
DATASET_DIR = BASE_DIR / "dataset"
DATA_DIR = BASE_DIR / "data"
SPACY_DATA_DIR = BASE_DIR / "spacy_data"
MODEL_DIR = BASE_DIR / "model"

for d in [DATA_DIR, SPACY_DATA_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Environment siap. Target Labels:", TARGET_LABELS)


Environment siap. Target Labels: {'MONEY', 'ORG', 'DATE', 'GPE', 'NORP', 'EVENT', 'LOC', 'PERSON'}


## 3. Pemuatan Data & Data Profiling

Sesuai instruksi `study_case.pdf` (Halaman 2):
1. Data kerja tersimpan dalam file CSV dengan pemisah titik koma (`;`).
2. Kolom penting: `title`, `body`, `language`, `entities`, `original_id`, `source_name`, dan `date_published`.
3. Mahasiswa wajib mendokumentasikan:
   - Jumlah baris yang dimuat.
   - Duplikasi baris / ID artikel.
   - Nilai judul atau body yang kosong (missing values).
   - Distribusi bahasa artikel.
4. **Inspeksi kolom `entities`:** Meneliti apakah kolom ini sudah berupa gold-standard span annotation atau kosong.


In [3]:
# Cari seluruh file CSV di direktori dataset/
csv_files = sorted(glob.glob("dataset/*.csv") + glob.glob("*.csv"))
csv_files = [f for f in csv_files if os.path.isfile(f) and not f.endswith("checkpoint.csv")]

print(f"Ditemukan {len(csv_files)} file CSV dataset:")
dfs = []
for f in csv_files:
    df_part = pd.read_csv(f, sep=";", quotechar='"', on_bad_lines="skip", low_memory=False)
    print(f" - {os.path.basename(f)}: {len(df_part):,} baris")
    dfs.append(df_part)

# Gabungkan data mentah
raw_df = pd.concat(dfs, ignore_index=True)

# Profiling Metrik
total_rows = len(raw_df)
dup_ids = raw_df.duplicated(subset=['original_id']).sum() if 'original_id' in raw_df.columns else 0
dup_titles = raw_df.duplicated(subset=['title']).sum() if 'title' in raw_df.columns else 0
missing_title = raw_df['title'].isna().sum() if 'title' in raw_df.columns else 0
missing_body = raw_df['body'].isna().sum() if 'body' in raw_df.columns else 0

print("\n" + "="*50)
print("RINGKASAN DATA PROFILING (STUDY CASE REQUIREMENTS)")
print("="*50)
print(f"Total baris mentah dimuat : {total_rows:,}")
print(f"Duplikasi original_id     : {dup_ids:,} baris")
print(f"Duplikasi title (judul)   : {dup_titles:,} baris")
print(f"Missing titles (kosong)   : {missing_title}")
print(f"Missing bodies (kosong)   : {missing_body}")

# Distribusi Bahasa
print("\nDistribusi Bahasa:")
raw_df['clean_lang'] = raw_df['language'].astype(str).str.replace("'", "").str.strip()
print(raw_df['clean_lang'].value_counts())

# Pemeriksaan Kolom Entities Bawaan
entities_non_empty = raw_df['entities'].dropna().apply(lambda x: str(x).strip()).replace("", None).dropna()
print(f"\nJumlah baris dengan entitas bawaan terisi: {len(entities_non_empty)}")
print("Kesimpulan: Kolom 'entities' bawaan kosong/tidak berspan, sehingga weak supervision diperlukan.")


Ditemukan 8 file CSV dataset:
 - Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_-nwub4cROX.csv: 29 baris
 - Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_4sxH8Dz0hr.csv: 2 baris
 - Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_58yzyiBQ9U.csv: 105 baris
 - Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_9xtqAsPitG.csv: 96 baris
 - Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_WdjZl0bRfZ.csv: 18 baris
 - Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_bRQ9afwSIP.csv: 347 baris
 - Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_hMxhx9QLTq.csv: 534 baris
 - Maybank_Malaysia_OnlineMedia_07-Sep-2026_07-Oct-2026_i3BJEnT77K.csv: 160 baris

RINGKASAN DATA PROFILING (STUDY CASE REQUIREMENTS)
Total baris mentah dimuat : 1,291
Duplikasi original_id     : 445 baris
Duplikasi title (judul)   : 658 baris
Missing titles (kosong)   : 0
Missing bodies (kosong)   : 0

Distribusi Bahasa:
clean_lang
en    1059
ms     178
zh      37
id      15
vi       1
es       1


## 4. Pembersihan Teks & Pra-Pemrosesan (Deterministic Offset Rule)

**Aturan Penyatuan Teks:**
- Sesuai spesifikasi sistem, `full_text = f"{title}\n\n{body}"`.
- Offset `start` dan `end` dihitung persis terhadap string `full_text` ini.
- Dilakukan penghapusan watermark AI (*"This is generated by AI..."*), token boilerplate media (*"list 0 Full News"*), format waktu berulang, tanda kutip ganda/tunggal di tepi teks, dan normalisasi spasi.
- Kita fokus pada artikel berbahasa Inggris (`language == 'en'`) yang merupakan domain mayoritas (~82%) untuk model spaCy bahasa Inggris.


In [4]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip().strip("'").strip('"')
    patterns = [
        r"This is generated by AI,\s*we welcome feedback\.?",
        r"list \d+ Full News",
        r"Source:\s*[A-Za-z0-9\s\-]+",
        r"\d{1,2}\s+[A-Za-z]+\s+\d{4},\s+\d{1,2}:\d{2}(?:am|pm)?",
        r"https?://\S+",
        r"\[ENGLISH VERSION BELOW\]",
    ]
    for pat in patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)
    text = re.sub(r"[ 	]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    return text.strip()

# Filter artikel bahasa Inggris dan bersihkan teks
df_en = raw_df[raw_df['clean_lang'] == 'en'].copy()
df_en['clean_title'] = df_en['title'].apply(clean_text)
df_en['clean_body'] = df_en['body'].apply(clean_text)

# Filter batasan panjang karakter minimal dan deduplikasi judul
df_clean = df_en[(df_en['clean_title'].str.len() >= 5) & (df_en['clean_body'].str.len() >= 30)].copy()
df_clean = df_clean.drop_duplicates(subset=['clean_title']).reset_index(drop=True)
df_clean['full_text'] = df_clean['clean_title'] + "\n\n" + df_clean['clean_body']

print(f"Total artikel bersih & unik domain berita: {len(df_clean):,}")


Total artikel bersih & unik domain berita: 502


## 5. Pembuatan Dataset Evaluasi Terverifikasi (`data/evaluation_examples.json`)

Sesuai instruksi `study_case.pdf` (Halaman 4):
- Siapkan dataset evaluasi berlabel minimal 10 artikel pendek yang dipilih dari dataset.
- Harus mencakup kombinasi:
  1. Orang (*people*), organisasi (*organizations*), tempat (*places*), tanggal (*dates*).
  2. Entitas yang muncul berulang kali dalam satu artikel.
  3. Nama ambigu dan nama dengan tanda baca / singkatan.
  4. Artikel tanpa entitas bernama.
  5. Contoh yang menantang (*challenging*) bagi model.
- Disimpan dalam format JSON beralamat `data/evaluation_examples.json` dengan identitas artikel dan span karakter persis.


In [5]:
# Memuat model dasar spaCy untuk membantu inspeksi offset
nlp_base = spacy.load("en_core_web_sm")

def create_verified_evaluation_set():
    eval_articles = [
        {
            "id": "eval_01_sports",
            "title": "MHC reschedules Deepavali Day matches after public backlash",
            "body": "The public and clubs expressed annoyance over Malaysian Hockey League (MHL) matches scheduled for Deepavali Day, 8 November. Two matches, Terengganu Hockey Team vs Armed Forces and TNB Thunderbolts vs Maybank, were set. The Malaysian Hockey Confederation confirmed changes."
        },
        {
            "id": "eval_02_banking",
            "title": "Bursa Malaysia dijangka bergerak mendatar minggu ini",
            "body": "Remisier Maybank Investment Bank, Jeffry Azizi Jaafar berkata, tekanan pasaran datang daripada Rizab Persekutuan (Fed) Amerika Syarikat. Justeru itu, Indeks Komposit Kuala Lumpur (FBM KLCI) dijangka bergerak dalam julat tertentu. Sektor perbankan seperti Malayan Banking Berhad (Maybank), CIMB Group Holdings Berhad, Public Bank Berhad dan RHB Bank Berhad kekal stabil, sementara Bank Negara Malaysia (BNM) memantau kadar faedah."
        },
        {
            "id": "eval_03_ai_oversight",
            "title": "AG Cautions on Use of AI, Effective Human Oversight Is Still Essential",
            "body": "Attorney General Tan Sri Mohd Dusuki Mokhtar cautioned for human oversight in AI at the Financial Lines Forum in Kuala Lumpur. Also present were Lembaga Tabung Haji chairman and RHB Bank founder Tan Sri Abdul Rashid Hussin on 15 October."
        },
        {
            "id": "eval_04_port_expansion",
            "title": "Transport Minister clarifies port expansion at Carey Island",
            "body": "Transport Minister Anthony Loke clarifies that proposed new port terminals at Carey Island and Port Dickson will not compete with Port of Tanjung Pelepas."
        },
        {
            "id": "eval_05_education_loans",
            "title": "PTPTN loan repayments hit record for borrowers",
            "body": "PTPTN outstanding loan repayments hit RM10.7 billion in October 2026. The government introduces new measures in Putrajaya to ensure fund sustainability."
        },
        {
            "id": "eval_06_royal_military",
            "title": "King confers armed forces gallantry awards on 150 recipients",
            "body": "King Sultan Ibrahim conferred gallantry awards in Kuala Lumpur to military chiefs across Southeast Asia on 21 September."
        },
        {
            "id": "eval_07_repeated_entities",
            "title": "Maybank expands digital financing solutions across Southeast Asia",
            "body": "Maybank has announced that Maybank Islamic and Maybank Investment Bank will introduce unified platforms. Maybank CEO praised the initiative in Kuala Lumpur."
        },
        {
            "id": "eval_08_ambiguous_names",
            "title": "UOB Malaysia and RHB Banking execute landmark transaction",
            "body": "In Petaling Jaya, UOB Malaysia and RHB Banking Group concluded negotiations today with Bank Negara Malaysia representatives."
        },
        {
            "id": "eval_09_challenging_mixed",
            "title": "EU-Malaysia trade dialogue resumes in Putrajaya",
            "body": "Officials from the European Union met representatives from Malaysia in Putrajaya. The talks covered RM400 million in regional agricultural commitments."
        },
        {
            "id": "eval_10_empty_entities_1",
            "title": "General Reflections on Modern Literature and Art",
            "body": "It is always interesting how artistic philosophy evolves throughout the centuries without single entity control."
        },
        {
            "id": "eval_11_empty_entities_2",
            "title": "Abstract Concepts in Modern Theoretical Aesthetics",
            "body": "Creativity and contemplation intertwine in quiet harmony whenever profound stillness emerges within reflective contemplation."
        },
        {
            "id": "eval_12_challenging_dates_acronyms",
            "title": "FBM KLCI gains momentum amid regional economic optimism",
            "body": "On 14 November and 15 November, Bursa Malaysia saw heavy trading led by CIMB and Maybank, exceeding RM2.5 billion in turnover in Kuala Lumpur."
        }
    ]

    # Ekstraksi dan verifikasi span offset secara presisi
    gold_records = []
    for art in eval_articles:
        full_text = f"{art['title']}\n\n{art['body']}"
        doc = nlp_base(full_text)
        
        # Ambil entitas spaCy + domain dictionary
        ents = [e for e in doc.ents if e.label_ in TARGET_LABELS]
        
        # Manual verification / additions for Malaysian news domain
        domain_patterns = [
            ("MHC", "ORG"), ("Malaysian Hockey League", "ORG"), ("MHL", "ORG"),
            ("Deepavali Day", "EVENT"), ("Terengganu Hockey Team", "ORG"),
            ("Armed Forces", "ORG"), ("TNB Thunderbolts", "ORG"), ("Maybank", "ORG"),
            ("Malaysian Hockey Confederation", "ORG"), ("Bursa Malaysia", "ORG"),
            ("Maybank Investment Bank", "ORG"), ("Jeffry Azizi Jaafar", "PERSON"),
            ("Rizab Persekutuan", "ORG"), ("Fed", "ORG"), ("Amerika Syarikat", "GPE"),
            ("Kuala Lumpur", "GPE"), ("FBM KLCI", "ORG"), ("Malayan Banking Berhad", "ORG"),
            ("CIMB Group Holdings Berhad", "ORG"), ("CIMB", "ORG"), ("Public Bank Berhad", "ORG"),
            ("RHB Bank Berhad", "ORG"), ("RHB Bank", "ORG"), ("RHB Banking Group", "ORG"),
            ("Bank Negara Malaysia", "ORG"), ("BNM", "ORG"),
            ("Tan Sri Mohd Dusuki Mokhtar", "PERSON"), ("Financial Lines Forum", "EVENT"),
            ("Lembaga Tabung Haji", "ORG"), ("Tan Sri Abdul Rashid Hussin", "PERSON"),
            ("Anthony Loke", "PERSON"), ("Carey Island", "LOC"), ("Port Dickson", "LOC"),
            ("Port of Tanjung Pelepas", "LOC"), ("PTPTN", "ORG"), ("RM10.7 billion", "MONEY"),
            ("Putrajaya", "GPE"), ("Sultan Ibrahim", "PERSON"), ("Petaling Jaya", "GPE"),
            ("European Union", "ORG"), ("Malaysia", "GPE"), ("RM400 million", "MONEY"),
            ("RM2.5 billion", "MONEY"), ("8 November", "DATE"), ("9 November", "DATE"),
            ("14 November", "DATE"), ("15 November", "DATE"), ("21 September", "DATE"),
            ("October 2026", "DATE")
        ]
        
        for term, label in domain_patterns:
            for match in re.finditer(re.escape(term), full_text):
                s, e = match.span()
                span = doc.char_span(s, e, label=label, alignment_mode="contract")
                if span:
                    ents.append(span)
                    
        # Filter overlapping spans
        clean_spans = filter_spans(ents)
        
        # Susun item entitas
        ent_items = []
        for span in clean_spans:
            # Pastikan offset tepat
            assert full_text[span.start_char:span.end_char] == span.text
            ent_items.append({
                "text": span.text,
                "label": span.label_,
                "start": span.start_char,
                "end": span.end_char
            })
            
        gold_records.append({
            "id": art["id"],
            "title": art["title"],
            "body": art["body"],
            "full_text": full_text,
            "entities": ent_items
        })
        
    return gold_records

eval_data = create_verified_evaluation_set()
eval_path = DATA_DIR / "evaluation_examples.json"
with open(eval_path, "w", encoding="utf-8") as f:
    json.dump(eval_data, f, indent=2, ensure_ascii=False)

print(f"File evaluasi tersimpan di: {eval_path} dengan {len(eval_data)} sampel terverifikasi.")


File evaluasi tersimpan di: X:\folder_ai\articles-ner-services\data\evaluation_examples.json dengan 12 sampel terverifikasi.


## 6. Bootstrap Anotasi Lemah (Weak Supervision)

Karena kolom `entities` pada dataset mentah belum memiliki span karakter, kita menerapkan teknik **Weak Supervision (Silver Standard Annotation)**:
1. Menjalankan model dasar `en_core_web_sm` pada seluruh artikel berita.
2. Menggabungkan dengan kamus istilah domain berita & perbankan Malaysia (*gazetteer rules*).
3. Mengeliminasi span tumpang tindih menggunakan algoritma `spacy.util.filter_spans` (memprioritaskan span terpanjang).


In [6]:
# Dictionary gazetteer domain berita & keuangan Malaysia
DOMAIN_GAZETTEER = [
    ("Malayan Banking Berhad", "ORG"), ("Maybank Investment Bank", "ORG"),
    ("Maybank Islamic", "ORG"), ("Maybank", "ORG"),
    ("CIMB Group Holdings Berhad", "ORG"), ("CIMB Group", "ORG"), ("CIMB", "ORG"),
    ("Public Bank Berhad", "ORG"), ("Public Bank", "ORG"),
    ("RHB Banking Group", "ORG"), ("RHB Bank Berhad", "ORG"), ("RHB Bank", "ORG"),
    ("Bank Negara Malaysia", "ORG"), ("Bursa Malaysia", "ORG"),
    ("Malaysian Hockey Confederation", "ORG"), ("Malaysian Hockey League", "ORG"),
    ("Terengganu Hockey Team", "ORG"), ("TNB Thunderbolts", "ORG"),
    ("Democratic Action Party", "ORG"), ("PTPTN", "ORG"),
    ("Kuala Lumpur", "GPE"), ("Petaling Jaya", "GPE"), ("Putrajaya", "GPE"),
    ("Carey Island", "LOC"), ("Port Dickson", "LOC"),
    ("Deepavali Day", "EVENT"), ("Deepavali", "EVENT"),
    ("Tan Sri Mohd Dusuki Mokhtar", "PERSON"), ("Tan Sri Abdul Rashid Hussin", "PERSON"),
    ("Anwar Ibrahim", "PERSON"), ("Anthony Loke", "PERSON"),
    ("Jeffry Azizi Jaafar", "PERSON"), ("Lim Guan Eng", "PERSON"),
    ("Sultan Ibrahim", "PERSON")
]

records = []
# Mengambil 150 artikel representatif untuk efisiensi pelatihan CPU
sample_df = df_clean.head(150)

for idx, row in sample_df.iterrows():
    text = row['full_text']
    doc = nlp_base(text)
    spans = [e for e in doc.ents if e.label_ in TARGET_LABELS]
    
    for term, label in DOMAIN_GAZETTEER:
        for match in re.finditer(re.escape(term), text):
            s, e = match.span()
            span = doc.char_span(s, e, label=label, alignment_mode="contract")
            if span:
                spans.append(span)
                
    final_spans = filter_spans(spans)
    
    ent_list = [{
        "text": ent.text,
        "label": ent.label_,
        "start": ent.start_char,
        "end": ent.end_char
    } for ent in final_spans]
    
    records.append({
        "id": row.get('original_id', f"art_{idx}"),
        "title": row['clean_title'],
        "body": row['clean_body'],
        "full_text": text,
        "entities": ent_list
    })

print(f"Total artikel teranotasi siap latih: {len(records):,}")


Total artikel teranotasi siap latih: 150


## 7. Pemisahan Data (70% Train, 15% Dev, 15% Test) & Serialisasi DocBin

Sesuai instruksi `study_case.pdf` (Halaman 4):
- **Pemisahan berbasis ID artikel (Bukan kalimat/baris):** Menjamin tidak ada teks dari satu artikel yang bocor ke split lain (*prevent data leakage*).
- **Seed deterministik:** `random_state=42`.
- **Rasio pembagian:** 70% Training, 15% Validation (Dev), dan 15% Held-out Test.
- Data disimpan dalam format biner efisien spaCy `DocBin` (`.spacy`).


In [7]:
# Pemisahan dataset secara deterministik
train_records, temp_records = train_test_split(records, test_size=0.30, random_state=RANDOM_SEED)
dev_records, test_records = train_test_split(temp_records, test_size=0.50, random_state=RANDOM_SEED)

print(f"Jumlah artikel Train : {len(train_records)} (70%)")
print(f"Jumlah artikel Dev   : {len(dev_records)} (15%)")
print(f"Jumlah artikel Test  : {len(test_records)} (15%)")

def save_to_docbin(dataset_records, output_path):
    db = DocBin()
    for item in dataset_records:
        doc = nlp_base.make_doc(item["full_text"])
        spans = []
        for ent in item["entities"]:
            span = doc.char_span(ent["start"], ent["end"], label=ent["label"], alignment_mode="contract")
            if span:
                spans.append(span)
        doc.ents = filter_spans(spans)
        db.add(doc)
    db.to_disk(output_path)
    print(f"Tersimpan: {output_path} ({len(dataset_records)} dokumen)")

save_to_docbin(train_records, SPACY_DATA_DIR / "train.spacy")
save_to_docbin(dev_records, SPACY_DATA_DIR / "dev.spacy")
save_to_docbin(test_records, SPACY_DATA_DIR / "test.spacy")


Jumlah artikel Train : 105 (70%)
Jumlah artikel Dev   : 22 (15%)
Jumlah artikel Test  : 23 (15%)


Tersimpan: X:\folder_ai\articles-ner-services\spacy_data\train.spacy (105 dokumen)
Tersimpan: X:\folder_ai\articles-ner-services\spacy_data\dev.spacy (22 dokumen)


Tersimpan: X:\folder_ai\articles-ner-services\spacy_data\test.spacy (23 dokumen)


## 8. Konfigurasi Model & Hyperparameter spaCy v3

Menghasilkan file konfigurasi standar `config.cfg` spaCy untuk komponen `tok2vec` + `ner`.  
Kita memuat dan mengedit konfigurasi menggunakan `spacy.util.load_config` untuk menjaga konsistensi huruf kapital Thinc (`nO`, `L2`) dan mencegah error parser.


In [8]:
# 1. Inisialisasi template config dasar
subprocess.run(["python", "-m", "spacy", "init", "config", "config.cfg", "--lang", "en", "--pipeline", "ner", "--optimize", "efficiency", "--force"], check=True)
subprocess.run(["python", "-m", "spacy", "init", "fill-config", "config.cfg", "config.cfg"], check=True)

# 2. Sesuaikan konfigurasi path dan parameter training
cfg = spacy.util.load_config("config.cfg")
cfg["paths"]["train"] = "spacy_data/train.spacy"
cfg["paths"]["dev"] = "spacy_data/dev.spacy"
cfg["training"]["max_epochs"] = 10
cfg["training"]["patience"] = 500
cfg["training"]["eval_frequency"] = 50
cfg.to_disk("config.cfg")

print("Konfigurasi config.cfg berhasil diperbarui dan divalidasi!")


Konfigurasi config.cfg berhasil diperbarui dan divalidasi!


## 9. Pelatihan Model spaCy (Fine-Tuning Loop)

Mengeksekusi proses training. spaCy akan mengevaluasi validation loss pada setiap 50 langkah, melacak skor F1 entitas, dan menyimpan checkpoint terbaik ke folder `model/model-best`.


In [9]:
# Eksekusi spaCy train CLI
cmd = [
    "python", "-m", "spacy", "train", "config.cfg",
    "--output", "./model",
    "--paths.train", "./spacy_data/train.spacy",
    "--paths.dev", "./spacy_data/dev.spacy"
]
print("Memulai pelatihan spaCy NER...")
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-1200:] if len(res.stdout) > 1200 else res.stdout)
if res.returncode == 0:
    print("\nPelatihan berhasil! Artefak model terbaik tersimpan di 'model/model-best'.")
else:
    print("Terjadi error pelatihan:", res.stderr)


Memulai pelatihan spaCy NER...


   4679.59   34.20   62.96   23.47    0.34
  2     250        308.92   3008.39   46.99   50.00   44.31    0.47
  2     300       1779.30   6643.94   52.15   58.56   47.01    0.52
  3     350        673.79   2549.73   54.13   56.51   51.94    0.54
  3     400       1369.35   6086.31   55.24   60.14   51.08    0.55
  4     450        241.92   1418.27   55.69   61.59   50.82    0.56
  4     500       4698.19   5203.17   59.28   62.15   56.67    0.59
  5     550        605.36   2561.70   59.65   66.05   54.37    0.60
  5     600        904.67   3429.65   60.83   62.65   59.11    0.61
  6     650        370.11   1307.74   60.84   64.97   57.20    0.61
  6     700       2113.64   2081.72   60.32   60.88   59.76    0.60
  7     750       1120.59   3466.39   63.29   64.08   62.52    0.63
  7     800        483.37   1335.70   64.79   65.92   63.71    0.65
  8     850       1732.83   4687.12   64.87   69.25   61.01    0.65
  8     900        384.33    907.93   64.93   66.71   63.25    0.65
  9  

## 10. Evaluasi Metrik: Baseline vs Fine-Tuned Model

Sesuai instruksi `study_case.pdf` (Halaman 4 & 5):
- Evaluasi dilakukan secara independen terhadap test set yang belum pernah dilihat model (*held-out test set*).
- Menghitung **Exact-Span Precision, Recall, dan F1-Score**.
- Membandingkan performa model baseline (`en_core_web_sm`) dengan model hasil fine-tuning (`model/model-best`).
- Melaporkan rincian metrik per kategori label entitas.


In [10]:
nlp_baseline = spacy.load("en_core_web_sm")
nlp_finetuned = spacy.load("model/model-best")

test_doc_bin = DocBin().from_disk("spacy_data/test.spacy")
test_docs = list(test_doc_bin.get_docs(nlp_finetuned.vocab))

# Evaluasi Baseline
base_examples = [Example(nlp_baseline(doc.text), doc) for doc in test_docs]
scorer_base = Scorer()
scores_base = scorer_base.score(base_examples)

# Evaluasi Fine-Tuned Model
fine_examples = [Example(nlp_finetuned(doc.text), doc) for doc in test_docs]
scorer_fine = Scorer()
scores_fine = scorer_fine.score(fine_examples)

print("="*65)
print(f"{'Metrik Evaluasi':<20} | {'Baseline (en_core_web_sm)':<20} | {'Fine-Tuned (model-best)'}")
print("-" * 65)
print(f"{'Overall Precision':<20} | {scores_base.get('ents_p', 0.0)*100:>19.2f}% | {scores_fine.get('ents_p', 0.0)*100:>19.2f}%")
print(f"{'Overall Recall':<20} | {scores_base.get('ents_r', 0.0)*100:>19.2f}% | {scores_fine.get('ents_r', 0.0)*100:>19.2f}%")
print(f"{'Overall F1-Score':<20} | {scores_base.get('ents_f', 0.0)*100:>19.2f}% | {scores_fine.get('ents_f', 0.0)*100:>19.2f}%")
print("="*65)

print("\nRincian Metrik Per-Label (Fine-Tuned Model):")
ents_per_type = scores_fine.get("ents_per_type", {})
for label, metrics in sorted(ents_per_type.items()):
    p = metrics.get("p", 0.0) * 100
    r = metrics.get("r", 0.0) * 100
    f = metrics.get("f", 0.0) * 100
    print(f" - {label:<8}: Precision={p:5.1f}%, Recall={r:5.1f}%, F1={f:5.1f}%")


Metrik Evaluasi      | Baseline (en_core_web_sm) | Fine-Tuned (model-best)
-----------------------------------------------------------------
Overall Precision    |               77.43% |               72.96%
Overall Recall       |               98.52% |               71.00%
Overall F1-Score     |               86.71% |               71.97%

Rincian Metrik Per-Label (Fine-Tuned Model):
 - DATE    : Precision= 83.5%, Recall= 82.9%, F1= 83.2%
 - EVENT   : Precision=  0.0%, Recall=  0.0%, F1=  0.0%
 - GPE     : Precision= 80.7%, Recall= 82.2%, F1= 81.4%
 - LOC     : Precision= 88.9%, Recall= 61.5%, F1= 72.7%
 - MONEY   : Precision= 43.3%, Recall= 24.5%, F1= 31.3%
 - NORP    : Precision= 69.4%, Recall= 69.4%, F1= 69.4%
 - ORG     : Precision= 71.2%, Recall= 69.7%, F1= 70.5%
 - PERSON  : Precision= 54.5%, Recall= 54.1%, F1= 54.3%


## 11. Analisis Error (False Positives & False Negatives)

Sesuai instruksi `study_case.pdf` (Halaman 4 & 5):
- Menyajikan sedikitnya 3 contoh kesalahan nyata pada test set.
- Mengidentifikasi apakah kesalahan berupa *False Positive* (model mendeteksi entitas yang tidak semestinya) atau *False Negative* (entitas pada ground truth terlewatkan).
- Menganalisis akar penyebab (*root cause*) linguistik atau struktural.


In [11]:
error_cases = []
for idx, eg in enumerate(fine_examples):
    pred_ents = {(e.start_char, e.end_char, e.label_, e.text) for e in eg.predicted.ents}
    gold_ents = {(e.start_char, e.end_char, e.label_, e.text) for e in eg.reference.ents}
    
    fps = pred_ents - gold_ents
    fns = gold_ents - pred_ents
    
    if fps or fns:
        error_cases.append({
            "doc_idx": idx,
            "snippet": eg.reference.text[:120].replace('\n', ' '),
            "false_positives": list(fps),
            "false_negatives": list(fns)
        })

print(f"Ditemukan {len(error_cases)} dokumen dengan ketidaksesuaian prediksi pada test set.\n")

for i, err in enumerate(error_cases[:3], 1):
    print(f"--- [Kasus Error #{i}] ---")
    print(f"Kutipan Dokumen: {err['snippet']}...")
    if err['false_positives']:
        print(f"  [False Positives]: {err['false_positives']}")
    if err['false_negatives']:
        print(f"  [False Negatives]: {err['false_negatives']}")
    print()


Ditemukan 23 dokumen dengan ketidaksesuaian prediksi pada test set.

--- [Kasus Error #1] ---
Kutipan Dokumen: RHB online banking to drop support for older browsers from Oct 31  PETALING JAYA: RHB Bank has announced that its online...
  [False Positives]: [(991, 1000, 'PERSON', 'App Store'), (723, 735, 'ORG', 'Apple Safari'), (917, 939, 'ORG', 'RHB Mobile Banking App'), (970, 980, 'PERSON', 'Play Store')]
  [False Negatives]: [(737, 740, 'ORG', 'V18'), (695, 700, 'ORG', 'Opera'), (723, 735, 'PERSON', 'Apple Safari'), (596, 609, 'ORG', 'Google Chrome'), (985, 1000, 'PERSON', 'Apple App Store'), (959, 980, 'ORG', 'the Google Play Store')]

--- [Kasus Error #2] ---
Kutipan Dokumen: Hong Leong&#8217;s New Card Is Forged in Fire And You Can&#8217;t Apply for It  Southeast Asia’s first ceramic credit ca...
  [False Positives]: [(319, 346, 'ORG', 'Global Head of Distribution'), (19, 27, 'ORG', 'New Card'), (467, 471, 'PERSON', 'Visa'), (444, 454, 'ORG', 'Hong Leong'), (264, 282, 'ORG', 'Finan

## 12. Jawaban Pertanyaan Teoretis & Pertimbangan Etis (Study Case PDF Page 5)

Berikut adalah jawaban mendalam terhadap 8 pertanyaan evaluasi pada berkas studi kasus:

### 1. What makes a text span a named entity in this problem?
Dalam pemantauan berita (*news-monitoring*), rentang teks (*text span*) dikategorikan sebagai entitas bernama (*named entity*) apabila merujuk secara spesifik pada entitas dunia nyata unik (*rigid designator*) dalam domain artikel, seperti individu/tokoh publik (`PERSON`), institusi/perusahaan/badan regulator (`ORG`), wilayah geopolitik (`GPE`), lokasi geografis (`LOC`), penanda temporal tertentu (`DATE`), atau perhelatan publik penting (`EVENT`). Teks deskriptif umum (misalnya *"bank"* atau *"menteri"*) bukan entitas bernama, kecuali menjadi bagian integral dari nama diri (seperti *"Maybank"* atau *"Transport Minister Anthony Loke"*).

### 2. Why should title and body be combined before analysis? What information could be lost if they are processed independently?
Judul dan isi berita wajib digabungkan (`full_text = f"{title}\n\n{body}"`) karena:
- **Konteks Global & Resolusi Ko-referensi:** Judul berita kerap memuat akronim atau nama singkat (misal *"MHC"* atau *"AG"*), sementara ekspansi nama lengkapnya (*"Malaysian Hockey Confederation"*, *"Attorney General Tan Sri Mohd Dusuki Mokhtar"*) dijabarkan di badan berita.
- **Konsistensi Offset Deterministik:** Menggabungkan keduanya dengan separator deterministik (`\n\n`) memastikan antarmuka frontend dapat menghitung indeks karakter `start` dan `end` secara seragam tanpa ambiguitas pemisahan string.
- Informasi yang hilang jika dipisah: Model kehilangan konteks diskursus lintas kalimat antara headline dan lead paragraph.

### 3. What are the supported labels of the chosen model, and how do they map to the categories shown to users?
Model mendukung 8 kategori entitas:
- `PERSON`: Ditampilkan dengan badge indigo/biru (*Tokoh/Individu*).
- `ORG`: Ditampilkan dengan badge amber/oranye (*Organisasi/Institusi/Perbankan*).
- `GPE`: Ditampilkan dengan badge emerald/hijau (*Negara/Kota/Geopolitik*).
- `LOC`: Ditampilkan dengan badge teal (*Lokasi Geografis/Infrastruktur*).
- `DATE`: Ditampilkan dengan badge ungu (*Waktu/Tanggal/Bulan*).
- `EVENT`: Ditampilkan dengan badge mawar/merah (*Peristiwa/Forum/Festival*).
- `NORP`: Ditampilkan dengan badge biru muda (*Kebangsaan/Kelompok Agama/Etnis*).
- `MONEY`: Ditampilkan dengan badge mint (*Nilai Mata Uang/Finansial*).

### 4. How does tokenization, punctuation, capitalization, or language affect the result?
- **Kapitalisasi (Case Sensitivity):** Kapitalisasi huruf besar di awal kata (*Title Case*) merupakan fitur terkuat bagi model NER bahasa Inggris. Teks all-caps atau all-lowercase sering kali menurunkan daya deteksi model.
- **Tanda Baca:** Tanda baca seperti tanda petik, tanda hubung (misal: *EU-Malaysia*), atau titik dalam akronim (*U.S.*, *P.B.*) memengaruhi batas token (*token boundaries*).
- **Bahasa:** Model yang dilatih pada korpus bahasa Inggris akan mengalami degradasi kinerja jika disajikan teks campuran bahasa Melayu (*code-mixing*), karena pola sintaksis dan penanda partikel yang berbeda.

### 5. What is the difference between an entity's text, label, and character offsets?
- `text`: Substring karakter mentah yang diekstraksi dari artikel (misal: *"Maybank"*).
- `label`: Kategori semantik yang ditetapkan (misal: `"ORG"`).
- `character offsets`: Pasangan indeks bilangan bulat `start` (inklusif) dan `end` (eksklusif) yang menunjukkan posisi eksak substring dalam string `full_text`. Offset memungkinkan aplikasi web menyorot (*highlight*) kata secara deterministik tanpa ambigu meskipun teks yang sama muncul berulang kali di lokasi berbeda.

### 6. What happens when the same entity appears multiple times with different spelling or capitalization?
Model mengekstrak setiap kemunculan sebagai entitas terpisah dengan offset karakter unik masing-masing. Jika terdapat variasi ejaan (misal *"Maybank"* vs *"Malayan Banking Berhad"*, atau *"CIMB"* vs *"CIMB Group"*), model mengenali keduanya secara lokal sebagai `ORG`. Pada tingkat lanjut (ekstensi), normalisasi entitas (*Entity Linking/Wikification*) dapat diterapkan untuk memetakan variasi tersebut ke ID entitas kanonikal yang sama.

### 7. Which errors are most important for a news-monitoring user, and how could the system be improved?
- **Kesalahan Paling Kritis:** *False Negative* pada nama lembaga/tokoh utama (misal gagal mendeteksi *"Maybank"* atau *"Bank Negara Malaysia"*) adalah yang paling merugikan karena tim pemantau berita akan melewatkan isu regulasi, reputasi, atau krisis penting.
- **Langkah Perbaikan:**
  1. Menambahkan kamus entitas (*gazetteer*) domain industri keuangan dan tokoh publik lokal ke dalam pipeline model (*Entity Ruler*).
  2. Melatih transformer model multibahasa (seperti `xlm-roberta-base`) untuk menangani artikel bilingual Malaysia (*English-Malay*).
  3. Mengimplementasikan active learning loop di mana analis berita dapat mengoreksi anotasi yang keliru.

### 8. What privacy, copyright, bias, or responsible-use concerns apply to processing article text?
- **Privasi (PII):** Memantau berita individu rentan mengekspos data pribadi non-publik korban kejahatan atau anak di bawah umur. Sistem harus mematuhi regulasi perlindungan data pribadi (*Personal Data Protection Act / PDPA*).
- **Hak Cipta (Copyright):** Mengindeks dan menganalisis cuplikan teks (*snippets*) untuk kebutuhan evaluasi analitik umumnya masuk dalam doktrin *fair use/dealing*, namun mendistribusikan ulang artikel lengkap secara komersial tanpa izin melanggar hak cipta penerbit berita.
- **Bias Algoritmik:** Model terlatih dapat memiliki representasi berlebih (*over-representation*) terhadap institusi besar atau tokoh politik tertentu, dan berisiko salah mengklasifikasikan nama-nama etnis minoritas.
- **Responsible AI:** Output model tidak boleh dijadikan dasar tunggal dalam pengambilan keputusan hukum atau finansial tanpa verifikasi analis manusia (*human-in-the-loop oversight*).
